# Embedding-Based Dictionary Expansion: Innovation Seeds

This notebook is a Google Colab teaching demo for a finance/accounting NLP workshop.

The exercise is inspired by Li, Mai, Shen, and Yan (2021, RFS), **"Measuring Corporate Culture Using Machine Learning"**. It is **not** a full replication. Instead, it demonstrates the core idea:

1. Start with a small set of seed words for a concept, here **innovation**.
2. Train Word2Vec on a domain corpus, here 2024Q4 US earnings-call Q&A turns.
3. Find words close to the average seed vector.
4. Manually inspect candidate words.
5. Compare domain-trained Word2Vec with pre-trained GloVe.

## 1. Setup

This notebook imports all dependencies and helper modules directly from GitHub raw URLs for seamless Colab execution. No local file path detection or fallbacks.

In [ ]:
import sys
import subprocess
import requests
from io import StringIO
import importlib.util

# Install required packages
packages = ["pandas", "numpy<2", "gensim", "scipy<1.13", "nltk", "tqdm", "requests"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

# GitHub raw URL for helper module
GITHUB_REPO = "helenlu-vbs/NLP_LLM_for_Finance_and-Accounting_Research-Sheffield-"
GITHUB_BRANCH = "main"
GITHUB_BASE_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_BRANCH}/"

# Download and load prepare_w2v_demo_artifacts.py directly from GitHub
script_url = f"{GITHUB_BASE_URL}scripts/prepare_w2v_demo_artifacts.py"
print(f"Loading helper module from: {script_url}")

response = requests.get(script_url)
response.raise_for_status()

# Create a module spec and load it
spec = importlib.util.spec_from_loader("prepare_w2v_demo_artifacts", loader=None)
demo = importlib.util.module_from_spec(spec)
exec(response.text, demo.__dict__)
sys.modules["prepare_w2v_demo_artifacts"] = demo

import numpy as np
import pandas as pd
import random
from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser

random.seed(demo.SEED)
np.random.seed(demo.SEED)

print("Using Word2Vec config:", demo.W2V_CONFIG)

## 2. Load Data

We load `data/raw/003_all_US_calls_2024Q4_top500.csv and data/raw/003_all_US_calls_2024Q4_other.csv` directly from GitHub raw content. The code automatically detects the main text column from a list of known candidates:

`["text", "content", "transcript", "componenttext", "speech", "turn_text", "qa_text", "sentence"]`

If none are found, it prints all columns and asks you to set `TEXT_COL` manually.

**GitHub note:** the notebook pulls all data directly from GitHub raw URLs for seamless Colab execution.

In [ ]:
# GitHub raw URLs for the CSV files
raw_urls = {
    "top500": f"{GITHUB_BASE_URL}data/raw/003_all_US_calls_2024Q4_top500.csv",
    "other": f"{GITHUB_BASE_URL}data/raw/003_all_US_calls_2024Q4_other.csv"
}

print("GitHub raw URLs:")
for name, url in raw_urls.items():
    print(f"  {name}: {url}")

# Function to load CSV from GitHub raw URL
def load_csv_from_github(url):
    print(f"Loading from: {url}")
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return pd.read_csv(StringIO(response.text))

# Load both CSV files
dfs = []
for name, url in raw_urls.items():
    df_temp = load_csv_from_github(url)
    print(f"  Loaded {name}: {df_temp.shape}")
    dfs.append(df_temp)

df = pd.concat(dfs, ignore_index=True)
print(f"\nCombined DataFrame shape: {df.shape}")

# Auto-detect text column
TEXT_COL = None  # set manually here if auto-detection fails, e.g. TEXT_COL = "turn_text"
df, text_col = demo.load_qna_data(None, text_col=TEXT_COL, df=df)

print("Shape after dropping missing/short text:", df.shape)
print("Detected text column:", text_col)
print("Columns:", list(df.columns))

for col in ["companyid", "companyname", "transcriptid", "speaker_name", "speaker_type"]:
    if col in df.columns:
        print(f"Unique {col}: {df[col].nunique():,}")

df[[c for c in ["companyname", "transcriptid", "speaker_name", "speaker_type", text_col] if c in df.columns]].head()

## 3. Lightweight Cleaning for Teaching

The full Li et al. pipeline uses Stanford CoreNLP, lemmatisation, NER replacement, dependency parsing for multiword expressions and compounds, stopword removal, Gensim phrase detection, and more.

For this classroom demo, we use a lightweight tokenizer that:

- lowercases text
- lightly expands common contractions
- keeps alphabetic tokens and selected hyphenated terms such as `ai-driven`
- removes punctuation and pure numbers
- removes one-letter tokens
- removes common English stopwords
- keeps finance-relevant words such as `may`, `growth`, `margin`, `cash`, `risk`

**Important:** these are Word2Vec-style tokens: words or phrase tokens. They are not LLM subword tokens.

In [ ]:
# The function is defined in the helper module so all cleaning logic is consistent.
clean_and_tokenize = demo.clean_and_tokenize

example_text = df[text_col].iloc[0]
print("Original text snippet:")
print(example_text[:500])
print("\nCleaned tokens:")
print(clean_and_tokenize(example_text)[:80])

## 4. Phrase Detection

We train Gensim `Phrases` and `Phraser` objects to create phrase-aware tokens such as:

- `customer_experience`
- `cash_flow`
- `gross_margin`

Defaults: `min_count=10`, `threshold=10`.

The notebook caches processed sentence files in temporary storage:

- `data/processed/w2v_sentences_unigram.txt`
- `data/processed/w2v_sentences_trigram.txt`

If these files already exist, they are loaded instead of rebuilt.

In [ ]:
from pathlib import Path
import tempfile

# Use temporary directory for caching
temp_dir = Path(tempfile.gettempdir()) / "w2v_demo"
temp_dir.mkdir(exist_ok=True)

unigram_path = temp_dir / "w2v_sentences_unigram.txt"
trigram_path = temp_dir / "w2v_sentences_trigram.txt"

if unigram_path.exists() and trigram_path.exists():
    print("Loading cached tokenized sentences.")
    tokenized = demo.load_sentences(unigram_path)
else:
    tokenized = demo.tokenize_texts(df[text_col])

unigram_sentences, trigram_sentences, bigram, trigram = demo.prepare_phrase_sentences(
    tokenized, temp_dir, force=False
)

print("Unigram turns:", len(unigram_sentences))
print("Trigram/phrase-aware turns:", len(trigram_sentences))
print("Example phrase-aware tokens:")
print(trigram_sentences[0][:80])

## 5. Train Word2Vec

For this workshop demo, we use the settings requested for a quick Colab run:

- `vector_size=50`
- `window=5`
- `min_count=10`
- `sg=1` (skip-gram)
- `negative=5`
- `epochs=3`
- `seed=42`

If the model files already exist, the notebook loads them instead of retraining:

- `models/w2v_2024q4_qna_demo.model`
- `models/w2v_2024q4_qna_demo.kv`

In [ ]:
model = demo.train_or_load_w2v(trigram_sentences, temp_dir, force=False)

total_tokens = sum(len(sent) for sent in trigram_sentences)
print("Number of sentences/turns:", f"{len(trigram_sentences):,}")
print("Total tokens:", f"{total_tokens:,}")
print("Vocabulary size:", f"{len(model.wv):,}")

## 6. Seed Words

We begin with a broader Li-style innovation seed list:

In [ ]:
li_innovation_seeds = [
    "innovation", "innovate", "innovative",
    "creativity", "creative", "create",
    "passion", "passionate",
    "efficiency", "efficient",
    "excellence", "pride",
]

# Use this broader Li-style seed list for the demo.
demo.INNOVATION_SEEDS = li_innovation_seeds
innovation_seeds = li_innovation_seeds

covered, missing = demo.covered_seeds(model.wv, innovation_seeds)
print("Covered seeds:", covered)
print("Missing seeds:", missing)
if len(covered) < 3:
    print("WARNING: fewer than 3 seeds are covered; continuing anyway.")

## 7. Nearest Neighbours in Domain-Trained Word2Vec

We average the vectors for the covered innovation seed words, compute cosine similarity to every vocabulary term, exclude the seeds themselves, and save the top 20 neighbours.

In [ ]:
w2v_top20 = demo.save_word2vec_neighbors(model, temp_dir)

display(w2v_top20)
print("Saved to temporary directory:")
print(temp_dir / "outputs/embedding_demo/innovation_top20_word2vec.csv")

## 8. Compare with Pre-Trained GloVe

GloVe knows general English from a broad external corpus. We use the same innovation seed list and the same average-seed-vector method.

If the download fails, the section prints a clear message and can be skipped in class.

In [ ]:
glove_top20 = demo.save_glove_neighbors(temp_dir)

if glove_top20 is not None:
    display(glove_top20)
else:
    print("GloVe download failed. This section can be skipped in class or run when internet is available.")

## 9. Compare with the Final Li-Style Innovation Dictionary

We load the reference dictionary from GitHub raw and compare top-20 Word2Vec and GloVe neighbours against it.

In [ ]:
# Load reference dictionary from GitHub raw
dict_url = f"{GITHUB_BASE_URL}data/processed/li_innovation_final_dictionary.txt"
print(f"Loading reference dictionary from: {dict_url}")

response = requests.get(dict_url)
response.raise_for_status()
li_innovation_reference = response.text.strip().split('\n')

print(f"Reference terms: {len(li_innovation_reference)}")
print(li_innovation_reference[:40])

overlap = demo.save_overlap_comparison(temp_dir, w2v_top20, glove_top20)
display(overlap)
print("Saved to temporary directory:")
print(temp_dir / "outputs/embedding_demo/innovation_overlap_comparison.csv")
print(temp_dir / "outputs/embedding_demo/innovation_li_final_dictionary_hit_comparison.csv")

## 10. Teaching Interpretation

Key points for students:

- **GloVe knows general English.** It was trained on broad, external text and often returns semantically general neighbours.
- **Word2Vec trained on earnings calls learns earnings-call language.** With the broader Li-style innovation seeds, the local Word2Vec model surfaces domain/business terms such as `customer_experience`, `platform`, `organic_growth`, which GloVe often misses.
- **In this split full-sample teaching corpus, Word2Vec now looks better for Li-style innovation expansion than GloVe.** In the tested run, Word2Vec top-20 has stronger overlap with the reference dictionary.
- **Seed choice matters.** The narrow seed list made GloVe look cleaner. The broader Li-style seeds anchor the domain-trained model better and produce more useful business-context candidates.
- **This does not mean the local Word2Vec output is a final dictionary.** Some candidates are still generic or noisy (`effort`, `supporting`, `merchandising`), so manual inspection is necessary.
- **A small one-quarter corpus is enough for demonstration, not publication.** A publishable dictionary would need a larger corpus, stronger preprocessing, validation, and documentation.
- **This differs from Li et al.** The full method uses heavier preprocessing, including Stanford CoreNLP, lemmatisation, NER replacement, dependency parsing, compound handling, stopword removal, and phrase detection with higher thresholds.

## 11. Optional Scoring Exercise

Students can choose 10–20 accepted innovation words after manual inspection of the top 20 neighbours. The simple score below counts accepted words divided by total tokens in each Q&A turn.

In [ ]:
# Edit this list after inspecting w2v_top20 and/or glove_top20.
accepted_innovation_words = [
    "innovation",
    "innovative",
    "creative",
    "technology",
    "platform",
    "automation",
    "workflow",
    "scalable",
    "efficient",
    "efficiency",
]

scores_by_firm = demo.optional_scoring_exercise(
    df, text_col, temp_dir, accepted_words=accepted_innovation_words
)

display(scores_by_firm.head(20))
print("Saved to temporary directory:")
print(temp_dir / "outputs/embedding_demo/simple_innovation_scores_by_firm.csv")